# LSTM — Long Short-Term Memory

> **Cheat Sheet Sections covered:** ⑪ Recurrent Neural Networks (RNN) · ⑫ Long Short-Term Memory (LSTM)

Standard feedforward networks ignore the **order** of inputs. RNNs pass a hidden state from step to step, but suffer from the **vanishing gradient** problem over long sequences. **LSTMs** solve this with a gated memory cell.

```mermaid
flowchart LR
    X["xₜ
(input)"] --> LSTM["LSTM Cell"]
    H["hₜ₋₁
(hidden state)"] --> LSTM
    C["cₜ₋₁
(cell state)"] --> LSTM
    LSTM --> Ho["hₜ
(output)"]
    LSTM --> Co["cₜ
(cell state)"]

    subgraph LSTM["LSTM Cell"]
        F["Forget Gate
fₜ = σ(Wf·[hₜ₋₁,xₜ]+bf)"]
        I["Input Gate
iₜ = σ(Wi·[hₜ₋₁,xₜ]+bi)"]
        G["Cell Candidate
g̃ₜ = tanh(Wg·[hₜ₋₁,xₜ]+bg)"]
        O["Output Gate
oₜ = σ(Wo·[hₜ₋₁,xₜ]+bo)"]
    end
```


## 1. Vanilla RNN — and Why It Fails

An RNN computes:  $h_t = \tanh(W_h h_{t-1} + W_x x_t + b)$

The hidden state $h_t$ is the only memory passed forward. When sequences are long, gradients during backpropagation must be multiplied through many timesteps → **vanishing** (or exploding) gradients.


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

# Demonstrate vanishing gradient in plain RNN
class VanillaRNNCell(nn.Module):
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.Wh = nn.Linear(hidden_size, hidden_size, bias=False)
        self.Wx = nn.Linear(input_size,  hidden_size, bias=True)

    def forward(self, x, h):
        return torch.tanh(self.Wh(h) + self.Wx(x))

rnn_cell = VanillaRNNCell(1, 16)
T = 60
x_seq = torch.randn(T, 1, 1)   # (T, batch=1, features=1)
h = torch.zeros(1, 16, requires_grad=True)

hs = []
for t in range(T):
    h = rnn_cell(x_seq[t], h)
    hs.append(h)

loss = hs[-1].sum()
loss.backward()

grad_norms = []
h_t = torch.zeros(1, 16, requires_grad=True)
for t in range(T):
    h_t = rnn_cell(x_seq[t], h_t)
    l   = h_t.sum()
    l.backward(retain_graph=True)
    if h_t.grad is not None:
        grad_norms.append(h_t.grad.norm().item())
    h_t = h_t.detach().requires_grad_(True)

plt.figure(figsize=(9, 4))
plt.plot(grad_norms[::-1], marker='o', markersize=3)
plt.xlabel('Timesteps back'); plt.ylabel('Gradient norm')
plt.title('Vanishing Gradient in Vanilla RNN — gradient shrinks over time')
plt.yscale('log'); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print(f"Gradient at t=1:    {grad_norms[0] if grad_norms else 'N/A':.6f}")
print(f"Gradient at t=-{T}: essentially zero → network cannot learn long-range dependencies")


## 2. LSTM Gates — Full Equations

The LSTM replaces the single hidden state with two: the **cell state** $c_t$ (long-term memory) and the **hidden state** $h_t$ (short-term / working memory).

| Gate | Equation | Role |
|---|---|---|
| **Forget gate** $f_t$ | $\sigma(W_f [h_{t-1}, x_t] + b_f)$ | What fraction of cell state to keep |
| **Input gate** $i_t$ | $\sigma(W_i [h_{t-1}, x_t] + b_i)$ | What new info to write |
| **Cell candidate** $\tilde{c}_t$ | $\tanh(W_c [h_{t-1}, x_t] + b_c)$ | Candidate values to write |
| **Cell update** $c_t$ | $f_t \odot c_{t-1} + i_t \odot \tilde{c}_t$ | Updated long-term memory |
| **Output gate** $o_t$ | $\sigma(W_o [h_{t-1}, x_t] + b_o)$ | What to expose as output |
| **Hidden state** $h_t$ | $o_t \odot \tanh(c_t)$ | Working memory / output |


In [ ]:
# LSTM cell from scratch to see the gates
class LSTMCellScratch(nn.Module):
    """LSTM cell implementing all four gates explicitly for pedagogical clarity"""
    def __init__(self, input_size, hidden_size):
        super().__init__()
        self.hidden_size = hidden_size
        # One combined linear for efficiency: [input, hidden] → [f, i, g, o]
        self.gates = nn.Linear(input_size + hidden_size, 4 * hidden_size)

    def forward(self, x, h_prev, c_prev):
        combined = torch.cat([x, h_prev], dim=1)
        gates    = self.gates(combined)         # (B, 4*H)

        f, i, g, o = gates.chunk(4, dim=1)     # split into 4 gates

        f = torch.sigmoid(f)                    # forget gate
        i = torch.sigmoid(i)                    # input gate
        g = torch.tanh(g)                       # cell candidate
        o = torch.sigmoid(o)                    # output gate

        c = f * c_prev + i * g                  # cell state update
        h = o * torch.tanh(c)                   # hidden state

        return h, c, {'f': f.detach(), 'i': i.detach(),
                      'g': g.detach(), 'o': o.detach()}

cell = LSTMCellScratch(input_size=4, hidden_size=8)
B, T = 2, 5

h = torch.zeros(B, 8)
c = torch.zeros(B, 8)

print("Stepping through an LSTM for 5 timesteps:\n")
for t in range(T):
    x = torch.randn(B, 4)
    h, c, gate_vals = cell(x, h, c)
    f_mean = gate_vals['f'].mean().item()
    i_mean = gate_vals['i'].mean().item()
    o_mean = gate_vals['o'].mean().item()
    print(f"  t={t+1}: forget={f_mean:.3f}  input={i_mean:.3f}  output={o_mean:.3f}  "
          f"h_norm={h.norm().item():.3f}  c_norm={c.norm().item():.3f}")

print("\n💡 Forget gate ≈1 → keep cell state  |  Input gate ≈1 → write new info")


## 3. PyTorch nn.LSTM

PyTorch's built-in `nn.LSTM` handles the full sequence loop for you.

```python
lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, dropout=p)
output, (h_n, c_n) = lstm(x, (h_0, c_0))
```

- `output` — hidden state at **every** timestep: shape `(B, T, hidden_size)`
- `h_n` — hidden state at the **last** timestep: shape `(num_layers, B, hidden_size)`  
- `c_n` — cell state at the **last** timestep


In [ ]:
# Compare nn.LSTM gradient flow vs plain RNN
lstm = nn.LSTM(input_size=1, hidden_size=16, batch_first=True)
rnn  = nn.RNN( input_size=1, hidden_size=16, batch_first=True)

T = 50
x_seq = torch.randn(1, T, 1, requires_grad=False)

# LSTM gradient norms
lstm_grads = []
for t in range(1, T+1):
    x_t = x_seq[:, :t, :].detach().requires_grad_(False)
    out, (h, c) = lstm(x_t)
    loss = out[:, -1, :].sum()
    loss.backward()
    g = sum(p.grad.norm().item() for p in lstm.parameters() if p.grad is not None)
    lstm_grads.append(g)
    lstm.zero_grad()

# RNN gradient norms
rnn_grads = []
for t in range(1, T+1):
    x_t = x_seq[:, :t, :].detach()
    out, h = rnn(x_t)
    loss = out[:, -1, :].sum()
    loss.backward()
    g = sum(p.grad.norm().item() for p in rnn.parameters() if p.grad is not None)
    rnn_grads.append(g)
    rnn.zero_grad()

plt.figure(figsize=(10, 4))
plt.plot(rnn_grads,  label='RNN',  linewidth=2)
plt.plot(lstm_grads, label='LSTM', linewidth=2)
plt.xlabel('Sequence length'); plt.ylabel('Total gradient norm')
plt.title('LSTM vs RNN: Gradient Flow Over Long Sequences')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()
print("LSTM maintains healthier gradients over long sequences.")


## 4. Sequence Classification with LSTM

Classifying whether a sequence is "increasing" or "decreasing" — a task that requires remembering the whole sequence.


In [ ]:
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim

torch.manual_seed(0); np.random.seed(0)

def make_seq_dataset(n=2000, seq_len=20):
    """Create sequences: class 0 = trending up, class 1 = trending down"""
    X, y = [], []
    for _ in range(n):
        label = np.random.randint(0, 2)
        trend = 1 if label == 0 else -1
        seq   = np.cumsum(trend * np.abs(np.random.randn(seq_len))) + np.random.randn(seq_len)*0.3
        X.append(seq); y.append(label)
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int64)

X, y = make_seq_dataset()
split = int(0.8 * len(X))
X_tr, X_te = X[:split], X[split:]
y_tr, y_te = y[:split], y[split:]

train_ld = DataLoader(TensorDataset(torch.FloatTensor(X_tr).unsqueeze(-1),
                                    torch.LongTensor(y_tr)), batch_size=64, shuffle=True)
test_ld  = DataLoader(TensorDataset(torch.FloatTensor(X_te).unsqueeze(-1),
                                    torch.LongTensor(y_te)), batch_size=256)

class LSTMClassifier(nn.Module):
    def __init__(self, input_size=1, hidden_size=32, num_layers=2, num_classes=2):
        super().__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                             batch_first=True, dropout=0.2)
        self.fc   = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        out, (h_n, _) = self.lstm(x)
        return self.fc(h_n[-1])      # use last layer's final hidden state

model    = LSTMClassifier()
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

train_accs, test_accs = [], []
for epoch in range(25):
    model.train()
    for bx, by in train_ld:
        out  = model(bx); loss = criterion(out, by)
        optimizer.zero_grad(); loss.backward(); optimizer.step()

    model.eval()
    def acc(loader):
        correct = total = 0
        with torch.no_grad():
            for bx, by in loader:
                preds = model(bx).argmax(1)
                correct += preds.eq(by).sum().item(); total += by.size(0)
        return 100.*correct/total

    tr_acc, te_acc = acc(train_ld), acc(test_ld)
    train_accs.append(tr_acc); test_accs.append(te_acc)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1:2d}/25  Train: {tr_acc:.1f}%  Test: {te_acc:.1f}%")

plt.figure(figsize=(9, 4))
plt.plot(train_accs, label='Train Accuracy', linewidth=2)
plt.plot(test_accs,  label='Test Accuracy',  linewidth=2)
plt.xlabel('Epoch'); plt.ylabel('Accuracy (%)')
plt.title('LSTM Sequence Classifier'); plt.legend(); plt.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()
print(f"\n✅ Final test accuracy: {test_accs[-1]:.1f}%")


## Summary

### ✅ What You Learned

1. **Vanilla RNN** — hidden state recurrence, vanishing gradient problem
2. **LSTM gates** — Forget / Input / Output gates, cell state vs hidden state
3. **nn.LSTM** — PyTorch API, `output` vs `h_n` vs `c_n`
4. **Sequence classification** — using the final hidden state for prediction

### Key Takeaway

The **cell state** $c_t$ acts as a conveyor belt — information can flow unchanged over many timesteps, protected by the forget gate. This is why LSTMs can learn long-range dependencies that vanilla RNNs cannot.

### 🎯 What's Next

**06_attention_mechanism** — attention further improves on LSTM by removing the sequential bottleneck entirely, allowing every token to attend directly to every other token.
